# ML-08 — Machine Learning Priority Scoring Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohila-Khaled-Abbas/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Training a Random Forest classifier on leak-free features to predict content refresh priorities.

## 1. Train Random Forest Classifier

*Train model on 80% client train set, evaluate on 20% client holdout set.*

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

csv_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
df["pos_clean"] = np.where(df["avg_position"] == 0, 50.0, df["avg_position"])
df["traffic_drop"] = np.maximum(0, df["clicks_prev_30d"] - df["clicks_last_30d"])
df["target_binary"] = (df["traffic_drop"] > df["traffic_drop"].median()).astype(int)

clients = df["client_id"].unique()
np.random.seed(42)
train_clients = np.random.choice(clients, size=int(len(clients)*0.8), replace=False)

train_df = df[df["client_id"].isin(train_clients)]
test_df = df[~df["client_id"].isin(train_clients)]

features = ["impressions_90d", "clicks_prev_30d", "pos_clean", "content_age_days", "ctr"]
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(train_df[features], train_df["target_binary"])

test_probs = rf.predict_proba(test_df[features])[:, 1]
auc_score = roc_auc_score(test_df["target_binary"], test_probs)
print(f"Holdout Client Test ROC-AUC: {auc_score:.4f}")


Holdout Client Test ROC-AUC: 0.9387


## 2. Feature Importances

*Extract feature importances from Random Forest model.*

In [2]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("=== FEATURE IMPORTANCES ===")
print(importances.to_string())


=== FEATURE IMPORTANCES ===
clicks_prev_30d     0.581236
ctr                 0.207092
impressions_90d     0.152298
pos_clean           0.034839
content_age_days    0.024535


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`